# Exploratory Data Analysis


In [4]:
import json

with open("../data/raw/NA1_5541699314.json", encoding='utf-8') as f:
    match = json.load(f)

In [5]:
for p in match['info']['participants']:
    print(p['teamId'], p['championName'], p['teamPosition'], p['win'])

100 Fiora TOP True
100 Elise JUNGLE True
100 Brand MIDDLE True
100 Samira BOTTOM True
100 Leona UTILITY True
200 Heimerdinger TOP False
200 Viego JUNGLE False
200 Lissandra MIDDLE False
200 Caitlyn BOTTOM False
200 Morgana UTILITY False


In [6]:
import pprint

match_row = {}

for participant in match["info"]["participants"]:
    prefix = "blue" if participant["teamId"] == 100 else "red"
    match_row[f"{prefix}_{participant['championName']}"] = 1

match_row["blue_win"] = int(match["info"]["teams"][0]["win"])

pprint.pprint(match_row)

{'blue_Brand': 1,
 'blue_Elise': 1,
 'blue_Fiora': 1,
 'blue_Leona': 1,
 'blue_Samira': 1,
 'blue_win': 1,
 'red_Caitlyn': 1,
 'red_Heimerdinger': 1,
 'red_Lissandra': 1,
 'red_Morgana': 1,
 'red_Viego': 1}


In [7]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.feature_engineering.dataset import create_pd_data_frame

df = create_pd_data_frame("../data/raw")

In [8]:
# Check the label balance — is it roughly 50/50?
print(df["blue_win"].value_counts())

# Check that each row has exactly 10 champions (5 blue + 5 red)
champ_cols = [c for c in df.columns if c != "blue_win"]
print(df[champ_cols].sum(axis=1).value_counts())

blue_win
1    3691
0    3594
Name: count, dtype: int64
10    7285
Name: count, dtype: int64


In [9]:
from src.feature_engineering.dataset import create_pd_data_frame

df = create_pd_data_frame('../data/raw')
y = df['blue_win']
x = df.drop(columns=['blue_win'])



In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# model = LogisticRegression(max_iter=1000)
model = LogisticRegression(max_iter=1000, C=0.01)
model.fit(X_train, y_train)
print(f'shape: {df.shape}')

print(f"Train accuracy: {model.score(X_train, y_train):.4f}")
print(f"Test accuracy: {model.score(X_test, y_test):.4f}")

shape: (7285, 345)
Train accuracy: 0.5975
Test accuracy: 0.4990


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"RF Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"RF Test accuracy: {rf.score(X_test, y_test):.4f}")

RF Train accuracy: 1.0000
RF Test accuracy: 0.4907


In [12]:
# Calculate each champion's overall win rate from your data
champ_wins = {}
champ_games = {}

for _, row in df.iterrows():
    for col in df.columns:
        if col == "blue_win":
            continue
        if row[col] == 1:
            champ_name = col  # e.g. "blue_Fiora"
            base_name = col.split("_", 1)[1]  # e.g. "Fiora"
            won = row["blue_win"] if col.startswith("blue") else 1 - row["blue_win"]
            champ_wins[base_name] = champ_wins.get(base_name, 0) + won
            champ_games[base_name] = champ_games.get(base_name, 0) + 1

win_rates = {champ: champ_wins[champ] / champ_games[champ] for champ in champ_games}

print(sorted(win_rates.items(), key=lambda x: x[1], reverse=True)[:10])

[('XinZhao', np.float64(0.5984848484848485)), ('Karthus', np.float64(0.5771812080536913)), ('Evelynn', np.float64(0.5685279187817259)), ('Rell', np.float64(0.5634517766497462)), ('Zeri', np.float64(0.5617283950617284)), ('Cassiopeia', np.float64(0.55)), ('Nilah', np.float64(0.5486111111111112)), ('Leblanc', np.float64(0.5481171548117155)), ('Olaf', np.float64(0.5474137931034483)), ('Kindred', np.float64(0.5462962962962963))]


In [13]:
def build_winrate_features(match_data: dict, win_rates: dict) -> dict:
    row = {}
    blue_rates = []
    red_rates = []

    
    for participant in match_data["info"]["participants"]:
        champ = participant["championName"]
        rate = win_rates.get(champ, 0.5)  # default to 50% if unseen
        if participant["teamId"] == 100:
            blue_rates.append(rate)
        else:
            red_rates.append(rate)
    
    row["blue_avg_winrate"] = sum(blue_rates) / len(blue_rates)
    row["red_avg_winrate"] = sum(red_rates) / len(red_rates)
    row["winrate_diff"] = row["blue_avg_winrate"] - row["red_avg_winrate"]
    row["blue_win"] = int(match_data["info"]["teams"][0]["win"])
    row["blue_max_winrate"] = max(blue_rates)
    row["blue_min_winrate"] = min(blue_rates)
    row["red_max_winrate"] = max(red_rates)
    row["red_min_winrate"] = min(red_rates)
    return row

In [14]:
import pandas as pd

rows = []
for file in Path("../data/raw").glob("*.json"):
    with open(file, encoding='utf-8') as f:
        match_data = json.load(f)
    if match_data.get("info", {}).get("teams"):
        rows.append(build_winrate_features(match_data, win_rates))

df2 = pd.DataFrame(rows)
X2 = df2.drop(columns=["blue_win"])
y2 = df2["blue_win"]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LogisticRegression(max_iter=1000)
model2.fit(X2_train, y2_train)
print(f"Train accuracy: {model2.score(X2_train, y2_train):.4f}")
print(f"Test accuracy: {model2.score(X2_test, y2_test):.4f}")

Train accuracy: 0.5786
Test accuracy: 0.5456


In [15]:
def build_combined_features(match_data: dict, win_rates: dict) -> dict | None:
    if not match_data.get("info", {}).get("teams"):
        return None
    
    row = {}
    blue_rates = []
    red_rates = []
    
    for participant in match_data["info"]["participants"]:
        champ = participant["championName"]
        prefix = "blue" if participant["teamId"] == 100 else "red"
        row[f"{prefix}_{champ}"] = 1
        
        rate = win_rates.get(champ, 0.5)
        if prefix == "blue":
            blue_rates.append(rate)
        else:
            red_rates.append(rate)
    
    row["blue_avg_winrate"] = sum(blue_rates) / len(blue_rates)
    row["red_avg_winrate"] = sum(red_rates) / len(red_rates)
    row["winrate_diff"] = row["blue_avg_winrate"] - row["red_avg_winrate"]
    row["blue_win"] = int(match_data["info"]["teams"][0]["win"])
    return row

In [16]:
rows = []
for file in Path("../data/raw").glob("*.json"):
    with open(file, encoding='utf-8') as f:
        match_data = json.load(f)
    result = build_combined_features(match_data, win_rates)
    if result is not None:
        rows.append(result)

df3 = pd.DataFrame(rows).fillna(0)
X3 = df3.drop(columns=["blue_win"])
y3 = df3["blue_win"]

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

# Logistic Regression
lr3 = LogisticRegression(max_iter=1000, C=0.01)
lr3.fit(X3_train, y3_train)
print(f"LR Train accuracy: {lr3.score(X3_train, y3_train):.4f}")
print(f"LR Test accuracy: {lr3.score(X3_test, y3_test):.4f}")

# Random Forest
rf3 = RandomForestClassifier(n_estimators=100, random_state=42)
rf3.fit(X3_train, y3_train)
print(f"RF Train accuracy: {rf3.score(X3_train, y3_train):.4f}")
print(f"RF Test accuracy: {rf3.score(X3_test, y3_test):.4f}")

LR Train accuracy: 0.5973
LR Test accuracy: 0.4997
RF Train accuracy: 1.0000
RF Test accuracy: 0.5271


In [17]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, max_depth=3, random_state=42)
xgb.fit(X2_train, y2_train)
print(f"XGB Train accuracy: {xgb.score(X2_train, y2_train):.4f}")
print(f"XGB Test accuracy: {xgb.score(X2_test, y2_test):.4f}")

XGB Train accuracy: 0.6917
XGB Test accuracy: 0.5319


In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path
import sys

import importlib
import src.data_collection.riot_client
importlib.reload(src.data_collection.riot_client)
from src.data_collection.riot_client import RiotClient

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_collection.riot_client import RiotClient

load_dotenv(override=True)
api_key = os.getenv("RIOT_API_KEY")
assert api_key is not None, "RIOT_API_KEY is not set"

client = RiotClient(api_key=api_key, region="americas", platform="na1")

puuid = client.get_puuid("Soup Lover", "ItLit")
assert puuid is not None, "Failed to resolve puuid"
print(f"PUUID: {puuid}")



rank_data = client.get_rank(puuid)

assert rank_data is not None, "Failed to get rank data"
print(rank_data)

# print(rank_data)

PUUID: MS13KwoaL7WrnivRLDGpJk05A3xauu6GTchzN1rZwIj5G8CBwHZOAKrOWe98X_D3LS3nRDOu8yS8EA
[{'leagueId': 'c3f4c6a5-52d4-4501-883e-f206fc11ab43', 'queueType': 'RANKED_SOLO_5x5', 'tier': 'PLATINUM', 'rank': 'III', 'puuid': 'MS13KwoaL7WrnivRLDGpJk05A3xauu6GTchzN1rZwIj5G8CBwHZOAKrOWe98X_D3LS3nRDOu8yS8EA', 'leaguePoints': 92, 'wins': 89, 'losses': 78, 'veteran': False, 'inactive': False, 'freshBlood': False, 'hotStreak': False}]
